# Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import optuna
import shap

import lightgbm as lgb

import xgboost as xgb

import umap
import hdbscan

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score

import sqlite3
import os

# Constants

In [ ]:
RANDOM_SEED = 42

# Inputs

In [19]:
DB_PATH = '../data/sqlite/data.db'

DATA_PATH = '../data/raw/CIC_IIoT_dataset_2025/generated_CSVs'

# Load Data

In [ ]:
conn = sqlite3.connect(DB_PATH)

# Check available tables
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("Available tables in the database:")
for table in tables:
    print(f"  • {table[0]}")

In [ ]:
def load_all_csvs_recursive(root_dir, pattern="*.csv", encoding=None, add_source_column=True, **read_csv_kwargs):
    """
    Load CSV files recursively from root_dir into a single DataFrame.

    Parameters
    ----------
    root_dir : str or Path
        Root directory to search recursively.
    pattern : str
        Filename glob pattern (default: "*.csv").
    encoding : str | None
        Optional encoding for pandas.read_csv.
    add_source_column : bool
        If True, adds a `__source_file__` column with file path.
    **read_csv_kwargs : dict
        Extra kwargs forwarded to pandas.read_csv.

    Returns
    -------
    pd.DataFrame
        Concatenated DataFrame from all found CSVs.
    """
    import fnmatch

    root_dir = os.path.abspath(root_dir)

    csv_paths = []
    for current_root, _, files in os.walk(root_dir):
        for file_name in files:
            if fnmatch.fnmatch(file_name, pattern):
                csv_paths.append(os.path.join(current_root, file_name))

    if not csv_paths:
        raise FileNotFoundError(f"No CSV files found recursively in: {root_dir} (pattern: {pattern})")

    frames = []
    failures = []

    for path in sorted(csv_paths):
        try:
            kwargs = dict(read_csv_kwargs)
            if encoding is not None:
                kwargs["encoding"] = encoding

            df_part = pd.read_csv(path, **kwargs)

            if add_source_column:
                df_part["__source_file__"] = path

            frames.append(df_part)
        except Exception as exc:
            failures.append((path, str(exc)))

    if not frames:
        raise ValueError("All CSV files failed to load.")

    merged_df = pd.concat(frames, ignore_index=True, sort=False)

    print(f"Loaded {len(frames)} CSV file(s)")
    print(f"Merged shape: {merged_df.shape}")

    if failures:
        print(f"Failed to load {len(failures)} file(s):")
        for file_path, err in failures[:10]:
            print(f"  - {file_path}: {err}")
        if len(failures) > 10:
            print(f"  ... and {len(failures) - 10} more")

    return merged_df


# Phase 1 - Binary Classification (Benign vs Malicious)

In [ ]:
df = pd.read_sql_query("SELECT * FROM CIC_IIoT_dataset_2025", conn)

In [ ]:
df

In [ ]:
y = df['label1']
X = df.drop(columns=df.select_dtypes(exclude='number').columns)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

In [ ]:
def objective(trial):
    params = {
        "objective": "multiclass",  # se for binário, troque para "binary"
        "num_class": y_train.nunique(),
        "metric": "multi_logloss",
        "boosting_type": "gbdt",
        "random_state": RANDOM_SEED,
        "n_estimators": tria    l.suggest_int("n_estimators", 200, 1200),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, 256),
        "max_depth": trial.suggest_int("max_depth", 3, 16),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }
    
    model = lgb.LGBMClassifier(**params)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1_weighted", n_jobs=-1)
    return scores.mean()

In [ ]:

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50, show_progress_bar=True)

In [ ]:
print("Best score:", study.best_value)
print("Best params:", study.best_params)

In [ ]:
#model.fit(X_train, y_train)
#y_pred = model.predict(X_test)
#print(classification_report(y_test, y_pred))

# Phase 2 - Multiclassification

# Phase 3 - Clustering